# Krycklan GLUE results by resolution

The notebook loads the newest timestamped run for each of 20, 40, 80 and 160 m. It uses the final CSV when available, otherwise the partial CSV, and reports runs that have not saved results yet. Check the timestamps and sample counts: the newest runs at different resolutions are not necessarily from the same calibration. Re-run the cells to refresh results while jobs are running.

The comparison covers all available resolutions. The detailed analysis below uses one selected resolution. This notebook does not run model simulations.

## Imports

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

load_dotenv()
sns.set_context('notebook', font_scale=1.5)

#%matplotlib widget
%load_ext autoreload
%autoreload 2

In [ ]:
results_folder = os.environ.get('SPAFHY_RESULTS_FOLDER')
if not results_folder or not results_folder.strip():
    raise EnvironmentError('SPAFHY_RESULTS_FOLDER is not set. Define it in .env or the environment.')
results_root = Path(results_folder).expanduser() / 'krycklan' / 'glue_krycklan'
resolutions = (20, 40, 80, 160)
soil_type = 'Till'
metrics = ['kge_tot', 'rmse_tot', 'mbe_tot', 'kge_sbsrf', 'rmse_sbsrf', 'mbe_sbsrf']
parameter_columns = [f'kmax_{soil_type}', f'kmin_{soil_type}', f'f_{soil_type}']

results_by_resolution = {}
run_info = []
for resolution in resolutions:
    resolution_dir = results_root / f'{resolution}m'
    run_dirs = sorted(
        path for path in resolution_dir.glob('*')
        if path.is_dir() and len(path.name) == 14 and path.name.isdigit()
    )
    if not run_dirs:
        print(f'{resolution} m: no timestamped run found in {resolution_dir}')
        continue
    run_dir = run_dirs[-1]
    final_csv = run_dir / 'glue_results.csv'
    partial_csv = run_dir / 'glue_results_partial.csv'
    if final_csv.is_file():
        csv_path, status = final_csv, 'final'
    elif partial_csv.is_file():
        csv_path, status = partial_csv, 'partial'
    else:
        print(f'{resolution} m: newest run {run_dir} has not saved results yet')
        continue

    frame = pd.read_csv(csv_path)
    missing = set(['sample_id', *parameter_columns, *metrics]) - set(frame.columns)
    if missing:
        raise ValueError(f'Missing columns in {csv_path}: {sorted(missing)}')
    if frame.empty:
        raise ValueError(f'No samples in {csv_path}; if it is being written, retry the cell')
    if frame['sample_id'].isna().any() or frame['sample_id'].duplicated().any():
        raise ValueError(f'sample_id values must be non-null and unique in {csv_path}')
    frame[parameter_columns + metrics] = frame[parameter_columns + metrics].apply(pd.to_numeric, errors='raise')
    if not np.isfinite(frame[parameter_columns + metrics].to_numpy()).all():
        raise ValueError(f'Parameter and metric values must be finite in {csv_path}')
    if (frame[[f'kmax_{soil_type}', f'kmin_{soil_type}']] <= 0).any().any():
        raise ValueError(f'kmax and kmin must be positive in {csv_path}')
    results_by_resolution[resolution] = frame.set_index('sample_id', drop=False)
    run_info.append({'resolution_m': resolution, 'run': run_dir.name, 'status': status,
                     'samples': len(frame), 'file': str(csv_path)})

if not results_by_resolution:
    raise FileNotFoundError(f'No GLUE results available under {results_root}')
display(pd.DataFrame(run_info))

## Compare resolutions

Histograms are normalized independently so runs with different sample counts can be compared. Stars mark the best KGE sample for each resolution and runoff component. Partial runs are only the samples completed so far.

In [ ]:
summary_rows = []
for resolution, frame in results_by_resolution.items():
    for suffix in ('tot', 'sbsrf'):
        best = frame.loc[frame[f'kge_{suffix}'].idxmax()]
        summary_rows.append({
            'resolution_m': resolution, 'runoff': suffix, 'samples': len(frame),
            'best_kge': best[f'kge_{suffix}'], 'best_kge_sample_id': best['sample_id'],
            'kmax_at_best_kge': best[f'kmax_{soil_type}'], 'f_at_best_kge': best[f'f_{soil_type}'],
            'min_rmse': frame[f'rmse_{suffix}'].min(),
        })
display(pd.DataFrame(summary_rows))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, suffix in zip(axes, ('tot', 'sbsrf')):
    for resolution, frame in results_by_resolution.items():
        sns.histplot(frame[f'kge_{suffix}'], bins=40, stat='density', element='step',
                     fill=False, ax=ax, label=f'{resolution} m (n={len(frame)})')
    ax.set_title(f'{suffix.upper()} KGE distributions')
    ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, len(results_by_resolution),
                         figsize=(5 * len(results_by_resolution), 9), squeeze=False,
                         sharex=True, sharey=True, layout='constrained')
for row, suffix in enumerate(('tot', 'sbsrf')):
    values = np.concatenate([frame[f'kge_{suffix}'].to_numpy() for frame in results_by_resolution.values()])
    norm = plt.Normalize(vmin=values.min(), vmax=values.max())
    for col, (resolution, frame) in enumerate(results_by_resolution.items()):
        ax = axes[row, col]
        scatter = ax.scatter(frame[f'kmax_{soil_type}'], frame[f'f_{soil_type}'],
                             c=frame[f'kge_{suffix}'], norm=norm, cmap='viridis', s=12, alpha=0.6)
        best = frame.loc[frame[f'kge_{suffix}'].idxmax()]
        #ax.scatter(best[f'kmax_{soil_type}'], best[f'f_{soil_type}'], marker='*',
        #           s=160, facecolors='red', edgecolors='black', label='Best KGE')
        ax.set_xscale('log')
        ax.set_title(f'{resolution} m - {suffix.upper()}')
        ax.set_xlabel('Kh$_{sat,max}$ (m s$^{-1}$)')
        ax.set_ylabel('f (m$^{-1}$)')
        #ax.legend()
    fig.colorbar(scatter, ax=axes[row, :].tolist(), label=f'KGE_{suffix}'.upper())
plt.show()

## Detailed analysis for one resolution

Change `detail_resolution` below to inspect another available resolution.

In [ ]:
detail_resolution = min(results_by_resolution)  # Replace with 20, 40, 80 or 160 to select explicitly.
if detail_resolution not in results_by_resolution:
    raise ValueError(f'No results loaded for {detail_resolution} m')
data = results_by_resolution[detail_resolution].copy()
print(f'Detailed analysis: {detail_resolution} m, {len(data)} samples')
print(f'Recorded kmin values: {data[f"kmin_{soil_type}"].unique()}')

# Calculate rank for each sample_id and metric

In [41]:
for metric_name in metrics:
    values = data[metric_name].abs() if metric_name.startswith('mbe_') else data[metric_name]
    ascending = not metric_name.startswith('kge_')
    data[f'rank_{metric_name}'] = values.rank(method='min', ascending=ascending)

In [ ]:
# Adjust these weights to change the importance of each metric in the final ranking.
w_kge = 1.0
w_rmse = 0.
w_mbe = 0.
N = len(data)
for suffix in ('tot', 'sbsrf'):
    data[f'points_{suffix}'] = (
        w_kge * (N - data[f'rank_kge_{suffix}'])
        + w_rmse * (N - data[f'rank_rmse_{suffix}'])
        + w_mbe * (N - data[f'rank_mbe_{suffix}'])
    )
    data[f'rank_{suffix}'] = data[f'points_{suffix}'].rank(method='min', ascending=False)

# GLUE analysis

Choose the number of simulations to retain for total and subsurface runoff
separately. Set one count to zero to skip that component; at least one must
be positive. Each set is ranked by KGE (highest first); the KGE of the last
retained simulation is shown as the threshold. Ties at the boundary are
resolved by CSV order. `color_metric` controls the parameter-plot colors.


In [ ]:
color_metric = 'kge'  # kge, rmse, mbe, or points; used for plot colors
n_sims_tot = 1
n_sims_sbsrf = 0
if color_metric not in ('kge', 'rmse', 'mbe', 'points'):
    raise ValueError(f'Unsupported color metric: {color_metric}')

for suffix, count in (('tot', n_sims_tot), ('sbsrf', n_sims_sbsrf)):
    if not isinstance(count, int) or isinstance(count, bool) or count < 0:
        raise ValueError(f'n_sims_{suffix} must be a non-negative integer')
    if count > len(data):
        raise ValueError(f'n_sims_{suffix}={count} exceeds the {len(data)} available samples')

if n_sims_tot == 0 and n_sims_sbsrf == 0:
    raise ValueError('At least one of n_sims_tot and n_sims_sbsrf must be positive')

kge_cutoff_tot = data['kge_tot'].nlargest(n_sims_tot, keep='first').iloc[-1] if n_sims_tot else None
kge_cutoff_sbsrf = data['kge_sbsrf'].nlargest(n_sims_sbsrf, keep='first').iloc[-1] if n_sims_sbsrf else None


In [ ]:
def metric_values(frame, metric_name, suffix):
    values = frame[f'{metric_name}_{suffix}']
    return values.abs() if metric_name == 'mbe' else values


fig, ax = plt.subplots(1, 2, figsize=(14, 6))
fig.subplots_adjust(wspace=0.3)
for idx, suffix in enumerate(['tot', 'sbsrf']):
    values = data[f'kge_{suffix}']
    cutoff = kge_cutoff_tot if suffix == 'tot' else kge_cutoff_sbsrf
    num_sims = n_sims_tot if suffix == 'tot' else n_sims_sbsrf
    sns.histplot(x=values, bins=100, kde=True, ax=ax[idx])
    if cutoff is not None:
        ax[idx].axvline(cutoff, color='red', linestyle='--', label=f'KGE threshold: {cutoff:.3f}')
    ax[idx].set_xlabel(f'kge_{suffix}')
    ax[idx].set_title(f'{suffix.upper()} KGE ({num_sims} simulations selected)')
    if cutoff is not None:
        ax[idx].legend()

In [ ]:
suitable_simulations_tot = data.nlargest(n_sims_tot, 'kge_tot', keep='first')
suitable_simulations_sbsrf = data.nlargest(n_sims_sbsrf, 'kge_sbsrf', keep='first')

for label, suitable, cutoff in (
    ('Total runoff', suitable_simulations_tot, kge_cutoff_tot),
    ('Subsurface runoff', suitable_simulations_sbsrf, kge_cutoff_sbsrf),
):
    if cutoff is None:
        print(f'{label}: skipped (0 simulations selected)')
    else:
        print(f'{label}: selected {len(suitable)} simulations; KGE threshold={cutoff:.3f}')

## Visualize sampled kmax and f

The top row shows simulations selected by the GLUE cutoff. The bottom row
shows all simulations over the full current GLUE search ranges: kmax from
1e-4 to 1e-2 m/s and f from 5 to 20. The red rectangle matches the shared
axis extent of the suitable simulations in the top row. A skipped component
has no top-row plot or extent rectangle. The current GLUE
runner fixes `kmin`; historical CSVs may contain varying `kmin` values.


In [27]:
print(f'kmax range in CSV: {data[f"kmax_{soil_type}"].min():.3g}–{data[f"kmax_{soil_type}"].max():.3g} m/s')
print(f'f range in CSV: {data[f"f_{soil_type}"].min():.3g}–{data[f"f_{soil_type}"].max():.3g}')

In [ ]:
F_BOUNDS = (5.0, 20.0)
KMAX_BOUNDS = (1e-4, 0.01)

fig, axes = plt.subplots(2, 2, figsize=(14, 12), sharex='row', sharey='row')
fig.subplots_adjust(wspace=0.4, hspace=0.3)

for row, row_axes in enumerate(axes):
    for ax, suitable, suffix in zip(
        row_axes, [suitable_simulations_tot, suitable_simulations_sbsrf], ['tot', 'sbsrf']
    ):
        full_range = row == 1
        if not full_range and suitable.empty:
            ax.set_axis_off()
            ax.set_title(f'{suffix.upper()} skipped (0 simulations selected)')
            continue
        plot_data = data if full_range else suitable
        colors = metric_values(plot_data, color_metric, suffix)
        scatter = ax.scatter(
            plot_data[f'kmax_{soil_type}'], plot_data[f'f_{soil_type}'],
            c=colors, cmap='viridis', s=20 if full_range else 50,
            alpha=0.25 if full_range else 0.6
        )
        ax.set_xscale('log')
        if full_range:
            f_padding = (F_BOUNDS[1] - F_BOUNDS[0]) * 0.05
            first_row_xlim = axes[0, 0].get_xlim()
            first_row_ylim = axes[0, 0].get_ylim()
            ax.set_xlim(
                min(KMAX_BOUNDS[0] / 10**0.05, first_row_xlim[0]),
                max(KMAX_BOUNDS[1] * 10**0.05, first_row_xlim[1]),
            )
            ax.set_ylim(
                min(F_BOUNDS[0] - f_padding, first_row_ylim[0]),
                max(F_BOUNDS[1] + f_padding, first_row_ylim[1]),
            )
            if not suitable.empty:
                ax.plot(
                    [first_row_xlim[0], first_row_xlim[1], first_row_xlim[1], first_row_xlim[0], first_row_xlim[0]],
                    [first_row_ylim[0], first_row_ylim[0], first_row_ylim[1], first_row_ylim[1], first_row_ylim[0]],
                    color='red', linestyle='--', linewidth=1.5, label='First-row extent'
                )
                ax.legend(loc='upper right')
            ax.set_title(f'{suffix.upper()} all simulations' + (' with suitable extent' if not suitable.empty else ''))
        else:
            ax.set_title(f'{suffix.upper()} suitable simulations (n={len(suitable)})')

        ax.set_xlabel('kmax (m/s)')
        ax.set_ylabel('f')
        color_label = f'abs(mbe_{suffix})' if color_metric == 'mbe' else f'{color_metric}_{suffix}'
        fig.colorbar(scatter, ax=ax, label=color_label)

## Visualize Ksat curves of suitable simulations

Using the current soil-generator defaults, conductivity is `kmax` down to
the constant surface depth, then decays as

$$K_{sat} = (K_{max} - K_{min})e^{-f(d-d_c)} + K_{min}, \quad d > d_c.$$

Here $d$ is positive depth and $d_c$ is the constant surface depth. Historical
results may have used different soil-generator settings.

In [ ]:
from scripts.create_soil_params import _CONST_SURF_KMAX, _MAX_DEPTH

constant_depth = _CONST_SURF_KMAX[soil_type]
depth = np.linspace(0, _MAX_DEPTH[soil_type], 100)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, suitable, title in zip(
    axes,
    [suitable_simulations_tot, suitable_simulations_sbsrf],
    ['TOT suitable simulations', 'SBSRF suitable simulations']
):
    if suitable.empty:
        ax.set_axis_off()
        ax.set_title(f'{title} (skipped)')
        continue
    for _, row in suitable.iterrows():
        kmin = row[f'kmin_{soil_type}']
        kmax = row[f'kmax_{soil_type}']
        f = row[f'f_{soil_type}']
        below_const = np.maximum(depth - constant_depth, 0)
        k_sat = (kmax - kmin) * np.exp(-f * below_const) + kmin
        ax.plot(np.log10(k_sat), -depth, color='grey', alpha=0.1)
    ax.set_xlabel('log10(k$_{sat}$ [m/s])')
    ax.set_ylabel('z (m)')
    ax.set_title(title)